In [1]:
from dqn import DQNAgent, DQNNetwork, play_episode, train
import gymnasium as gym
from visualization import show_animation
import os
import numpy as np


In [2]:
models_dir = "./models"

os.makedirs(models_dir, exist_ok=True)

policy_net_path = os.path.join(models_dir, "dqn_cartpole")

data_dir = "./data"
a2c_data_dir = os.path.join(data_dir, "dqn")
os.makedirs(a2c_data_dir, exist_ok=True)

x_path = os.path.join(a2c_data_dir, "X_cartpole.npy")
y_path = os.path.join(a2c_data_dir, "y_cartpole.npy")

print(policy_net_path)


./models\dqn_cartpole


In [3]:
env = gym.make('CartPole-v1', render_mode="rgb_array")


In [4]:
obs_size = env.observation_space.shape[0]
action_size = env.action_space.n

print(obs_size, action_size)


4 2


In [5]:
agent = DQNAgent(
    state_size=obs_size,
    action_size=action_size
)


In [6]:
env = gym.make("CartPole-v1", render_mode="rgb_array")


In [7]:
agent.load(policy_net_path)


In [8]:
def collect_dataset(env: gym.Env, agent: DQNAgent, n_episodes: int = 200):
	X, y = [], []
	
	for _ in range(n_episodes):
		state, _ = env.reset()
		done = False
		
		while not done:
			q_values = agent.predict(state)
			action = np.argmax(q_values)
			
			X.append(state)
			y.append(action)
			
			state, _, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
	return np.array(X), np.array(y)

X, y = collect_dataset(env, agent, n_episodes=100)

print("Dataset:", X.shape, y.shape)
print("Distribución de acciones:", np.bincount(y))


Dataset: (50000, 4) (50000,)
Distribución de acciones: [25000 25000]


In [9]:
np.save(x_path, X)
np.save(y_path, y)


In [10]:
episode_reward, episode_steps, frames = play_episode(env, agent)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")

env.close()


c:\Users\malos\Documents\GitHub\XRL\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Recompensa del episodio: 500.0
Pasos del episodio: 500


In [11]:
show_animation(frames)
